In [11]:
import numpy as np
import scipy.io as sio
import xarray as xr
from pathlib import Path
from tqdm.std import tqdm


# ============================================================
# Directional anisotropy metric
# ============================================================
def calculate_anisotropy_metric(
    directional_values,
    min_valid_orientations=3
):
    """
    Calculate directional variability:

        A(r) = std_theta[D(r, theta)] /
               (abs(mean_theta[D(r, theta)])
                + rms_theta[D(r, theta)])

    Parameters
    ----------
    directional_values : ndarray
        Shape (nscale, norientation).

    min_valid_orientations : int
        Minimum number of valid orientations.

    Returns
    -------
    dict
        A
        mean
        std
        rms
        cancellation_ratio
        nvalid_orientations
    """
    directional_values = np.asarray(
        directional_values,
        dtype=float
    )

    nscale = directional_values.shape[0]

    A = np.full(nscale, np.nan)
    mean_value = np.full(nscale, np.nan)
    std_value = np.full(nscale, np.nan)
    rms_value = np.full(nscale, np.nan)
    cancellation_ratio = np.full(nscale, np.nan)

    nvalid = np.zeros(
        nscale,
        dtype=int
    )

    for ir in range(nscale):

        values = directional_values[ir]

        values = values[
            np.isfinite(values)
        ]

        nvalid[ir] = values.size

        if values.size < min_valid_orientations:
            continue

        mean_value[ir] = np.mean(values)

        std_value[ir] = np.std(
            values,
            ddof=0
        )

        rms_value[ir] = np.sqrt(
            np.mean(values**2)
        )

        denominator = (
            np.abs(mean_value[ir])
            + rms_value[ir]
        )

        if denominator > 0:
            A[ir] = (
                std_value[ir]
                / denominator
            )

        if rms_value[ir] > 0:
            cancellation_ratio[ir] = (
                np.abs(mean_value[ir])
                / rms_value[ir]
            )

    return {
        "A": A,
        "mean": mean_value,
        "std": std_value,
        "rms": rms_value,
        "cancellation_ratio":
            cancellation_ratio,
        "nvalid_orientations": nvalid,
    }


# ============================================================
# Interpolate one orientation to common requested scales
# ============================================================
def interpolate_to_common_scales(
    r_actual,
    values,
    r_target
):
    """
    Interpolate one directional/orientation curve onto the
    common requested separation coordinates.

    Interpolation is linear in log(r).
    """
    r_actual = np.asarray(
        r_actual,
        dtype=float
    )

    values = np.asarray(
        values,
        dtype=float
    )

    r_target = np.asarray(
        r_target,
        dtype=float
    )

    valid = (
        np.isfinite(r_actual)
        & np.isfinite(values)
        & (r_actual > 0)
        & (r_target.min() > 0)
    )

    result = np.full(
        r_target.shape,
        np.nan
    )

    if np.count_nonzero(valid) < 2:
        return result

    x = r_actual[valid]
    y = values[valid]

    order = np.argsort(x)
    x = x[order]
    y = y[order]

    # Average repeated actual separation values
    x_unique = np.unique(x)
    y_unique = np.full(
        x_unique.shape,
        np.nan
    )

    for i, x_value in enumerate(x_unique):
        same = np.isclose(
            x,
            x_value,
            rtol=1e-10,
            atol=0
        )

        y_unique[i] = np.mean(
            y[same]
        )

    if x_unique.size < 2:
        return result

    inside = (
        (r_target >= x_unique.min())
        & (r_target <= x_unique.max())
    )

    result[inside] = np.interp(
        np.log(r_target[inside]),
        np.log(x_unique),
        y_unique
    )

    return result


# ============================================================
# Directional Eulerian structure functions
# ============================================================
def calculate_eulerian_directional_SF(
    u,
    v,
    r_values,
    dx,
    dy=None,
    periodic=False,
    directions=None,
    time_chunk=50,
    show_progress=True,
    progress_name="Directional Eulerian SF",
    min_valid_orientations=3
):
    """
    Calculate direction-dependent Eulerian structure functions.

    Components
    ----------
    Dl   = <du_L>
    Dt   = <du_T>
    D1   = Dl + Dt

    Dll  = <du_L^2>
    Dtt  = <du_T^2>
    D2   = Dll + Dtt

    Dlll = <du_L^3>
    Dltt = <du_L du_T^2>
    D3   = Dlll + Dltt

    Output contains:
        direction_*:
            Results for every signed direction.

        orientation_*:
            Results after combining opposite directions.

        A_*:
            Directional anisotropy metrics calculated across
            the independent orientations.
    """

    # --------------------------------------------------------
    # Input checks
    # --------------------------------------------------------
    u = np.asarray(u)
    v = np.asarray(v)

    if u.shape != v.shape:
        raise ValueError(
            "u and v must have identical shapes."
        )

    if u.ndim == 2:
        u = u[None, :, :]
        v = v[None, :, :]

    if u.ndim != 3:
        raise ValueError(
            "u and v must have shape "
            "(ntime, ny, nx) or (ny, nx)."
        )

    if dy is None:
        dy = dx

    r_values = np.asarray(
        r_values,
        dtype=float
    ).ravel()

    if np.any(r_values <= 0):
        raise ValueError(
            "All r_values must be positive."
        )

    # Eight signed directions
    if directions is None:
        directions = [
            (0, 1),     # east
            (1, 1),     # northeast
            (1, 0),     # north
            (1, -1),    # northwest
            (0, -1),    # west
            (-1, -1),   # southwest
            (-1, 0),    # south
            (-1, 1),    # southeast
        ]

    directions = np.asarray(
        directions,
        dtype=int
    )

    ntime, ny, nx = u.shape
    nscale = len(r_values)
    ndirection = len(directions)

    component_names = [
        "Dl",
        "Dt",
        "D1",
        "Dll",
        "Dtt",
        "D2",
        "Dlll",
        "Dltt",
        "D3",
    ]

    print(
        f"{progress_name}\n"
        f"  velocity shape: {u.shape}\n"
        f"  requested scales: {nscale}\n"
        f"  signed directions: {ndirection}\n"
        f"  time chunk: {time_chunk}\n"
        f"  periodic: {periodic}"
    )

    # --------------------------------------------------------
    # Direction angles
    # --------------------------------------------------------
    direction_angles = np.zeros(
        ndirection,
        dtype=float
    )

    for idir, (base_di, base_dj) in enumerate(
        directions
    ):
        direction_angles[idir] = (
            np.degrees(
                np.arctan2(
                    base_di * dy,
                    base_dj * dx
                )
            )
            % 360.0
        )

    # --------------------------------------------------------
    # Raw signed-direction storage
    # --------------------------------------------------------
    direction_values = {
        name: np.full(
            (nscale, ndirection),
            np.nan
        )
        for name in component_names
    }

    direction_count = np.zeros(
        (nscale, ndirection),
        dtype=np.int64
    )

    direction_r_actual = np.full(
        (nscale, ndirection),
        np.nan
    )

    # --------------------------------------------------------
    # Nonperiodic slicing helper
    # --------------------------------------------------------
    def paired_slices(size, shift):

        if shift > 0:
            reference = slice(0, size - shift)
            displaced = slice(shift, size)

        elif shift < 0:
            reference = slice(-shift, size)
            displaced = slice(0, size + shift)

        else:
            reference = slice(0, size)
            displaced = slice(0, size)

        return reference, displaced

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------
    nchunks = int(
        np.ceil(ntime / time_chunk)
    )

    total_steps = (
        nscale
        * ndirection
        * nchunks
    )

    if show_progress:
        progress = tqdm(
            total=total_steps,
            desc=progress_name,
            unit="chunk"
        )
    else:
        progress = None

    # ========================================================
    # Scale loop
    # ========================================================
    for ir, target_r in enumerate(r_values):

        # ====================================================
        # Direction loop
        # ====================================================
        for idir, (base_di, base_dj) in enumerate(
            directions
        ):

            base_distance = np.sqrt(
                (base_di * dy)**2
                + (base_dj * dx)**2
            )

            if base_distance == 0:
                if progress is not None:
                    progress.update(nchunks)
                continue

            multiplier = max(
                1,
                int(np.round(
                    target_r / base_distance
                ))
            )

            di = int(
                base_di * multiplier
            )

            dj = int(
                base_dj * multiplier
            )

            if not periodic:
                if abs(di) >= ny or abs(dj) >= nx:
                    if progress is not None:
                        progress.update(nchunks)
                    continue

            actual_r = np.sqrt(
                (di * dy)**2
                + (dj * dx)**2
            )

            if actual_r <= 0:
                if progress is not None:
                    progress.update(nchunks)
                continue

            direction_r_actual[
                ir,
                idir
            ] = actual_r

            # Longitudinal unit vector
            ex = dj * dx / actual_r
            ey = di * dy / actual_r

            # Transverse unit vector:
            # e_T = (-ey, ex)
            tx = -ey
            ty = ex

            component_sum = {
                name: 0.0
                for name in component_names
            }

            valid_count = 0

            if not periodic:
                y0, y1 = paired_slices(
                    ny,
                    di
                )

                x0, x1 = paired_slices(
                    nx,
                    dj
                )

            # ================================================
            # Time-chunk loop
            # ================================================
            for t0 in range(
                0,
                ntime,
                time_chunk
            ):
                t1 = min(
                    t0 + time_chunk,
                    ntime
                )

                u_chunk = u[t0:t1]
                v_chunk = v[t0:t1]

                if periodic:

                    u_displaced = np.roll(
                        u_chunk,
                        shift=(-di, -dj),
                        axis=(1, 2)
                    )

                    v_displaced = np.roll(
                        v_chunk,
                        shift=(-di, -dj),
                        axis=(1, 2)
                    )

                    du = (
                        u_displaced
                        - u_chunk
                    )

                    dv = (
                        v_displaced
                        - v_chunk
                    )

                else:

                    u_reference = u_chunk[
                        :,
                        y0,
                        x0
                    ]

                    u_displaced = u_chunk[
                        :,
                        y1,
                        x1
                    ]

                    v_reference = v_chunk[
                        :,
                        y0,
                        x0
                    ]

                    v_displaced = v_chunk[
                        :,
                        y1,
                        x1
                    ]

                    du = (
                        u_displaced
                        - u_reference
                    )

                    dv = (
                        v_displaced
                        - v_reference
                    )

                # Longitudinal and transverse increments
                du_L = (
                    du * ex
                    + dv * ey
                )

                du_T = (
                    du * tx
                    + dv * ty
                )

                L = du_L.ravel()
                T = du_T.ravel()

                valid = (
                    np.isfinite(L)
                    & np.isfinite(T)
                )

                L = L[valid]
                T = T[valid]

                if L.size > 0:

                    L2 = L * L
                    T2 = T * T
                    L3 = L2 * L
                    LTT = L * T2

                    component_sum["Dl"] += np.sum(L)
                    component_sum["Dt"] += np.sum(T)

                    component_sum["D1"] += np.sum(
                        L + T
                    )

                    component_sum["Dll"] += np.sum(L2)
                    component_sum["Dtt"] += np.sum(T2)

                    component_sum["D2"] += np.sum(
                        L2 + T2
                    )

                    component_sum["Dlll"] += np.sum(L3)

                    component_sum["Dltt"] += np.sum(
                        LTT
                    )

                    component_sum["D3"] += np.sum(
                        L3 + LTT
                    )

                    valid_count += L.size

                    del L2, T2, L3, LTT

                del du, dv
                del du_L, du_T
                del L, T, valid

                if periodic:
                    del u_displaced
                    del v_displaced
                else:
                    del u_reference
                    del u_displaced
                    del v_reference
                    del v_displaced

                if progress is not None:
                    progress.set_postfix_str(
                        f"r={target_r:.3g}, "
                        f"theta={direction_angles[idir]:.0f}"
                    )
                    progress.update(1)

            if valid_count > 0:

                direction_count[
                    ir,
                    idir
                ] = valid_count

                for name in component_names:

                    direction_values[name][
                        ir,
                        idir
                    ] = (
                        component_sum[name]
                        / valid_count
                    )

    if progress is not None:
        progress.close()

    # ========================================================
    # Combine opposite signed directions into orientations
    #
    # For default directions:
    #   east + west
    #   northeast + southwest
    #   north + south
    #   northwest + southeast
    # ========================================================
    orientation_groups = {}

    for idir, angle in enumerate(
        direction_angles
    ):

        orientation_angle = (
            angle % 180.0
        )

        # Avoid floating-point grouping problems
        key = round(
            orientation_angle,
            8
        )

        if key not in orientation_groups:
            orientation_groups[key] = []

        orientation_groups[key].append(
            idir
        )

    orientation_angles = np.array(
        sorted(orientation_groups.keys()),
        dtype=float
    )

    norientation = len(
        orientation_angles
    )

    orientation_values_raw = {
        name: np.full(
            (nscale, norientation),
            np.nan
        )
        for name in component_names
    }

    orientation_r_actual = np.full(
        (nscale, norientation),
        np.nan
    )

    orientation_count = np.zeros(
        (nscale, norientation),
        dtype=np.int64
    )

    # --------------------------------------------------------
    # Pair opposite directions using count weighting
    # --------------------------------------------------------
    for iorientation, angle in enumerate(
        orientation_angles
    ):

        members = orientation_groups[
            round(angle, 8)
        ]

        for ir in range(nscale):

            counts = direction_count[
                ir,
                members
            ].astype(float)

            valid_count = counts > 0

            if not np.any(valid_count):
                continue

            member_indices = np.asarray(
                members
            )[valid_count]

            weights = counts[
                valid_count
            ]

            orientation_count[
                ir,
                iorientation
            ] = int(
                np.sum(weights)
            )

            orientation_r_actual[
                ir,
                iorientation
            ] = np.average(
                direction_r_actual[
                    ir,
                    member_indices
                ],
                weights=weights
            )

            for name in component_names:

                values = direction_values[name][
                    ir,
                    member_indices
                ]

                valid_value = np.isfinite(
                    values
                )

                if np.any(valid_value):

                    orientation_values_raw[name][
                        ir,
                        iorientation
                    ] = np.average(
                        values[valid_value],
                        weights=weights[
                            valid_value
                        ]
                    )

    # ========================================================
    # Interpolate each orientation onto common target r
    # ========================================================
    orientation_values = {
        name: np.full(
            (nscale, norientation),
            np.nan
        )
        for name in component_names
    }

    for name in component_names:

        for iorientation in range(
            norientation
        ):

            orientation_values[name][
                :,
                iorientation
            ] = interpolate_to_common_scales(
                r_actual=orientation_r_actual[
                    :,
                    iorientation
                ],
                values=orientation_values_raw[
                    name
                ][
                    :,
                    iorientation
                ],
                r_target=r_values
            )

    # ========================================================
    # Calculate anisotropy metrics
    # ========================================================
    anisotropy = {}

    for name in component_names:

        anisotropy[name] = (
            calculate_anisotropy_metric(
                orientation_values[name],
                min_valid_orientations=
                    min_valid_orientations
            )
        )

    # ========================================================
    # Flat output dictionary
    # ========================================================
    result = {
        "r_requested": r_values,
        "direction_vectors": directions,
        "direction_angles_deg":
            direction_angles,
        "direction_r_actual":
            direction_r_actual,
        "direction_count":
            direction_count,
        "orientation_angles_deg":
            orientation_angles,
        "orientation_r_actual":
            orientation_r_actual,
        "orientation_count":
            orientation_count,
        "periodic": periodic,
        "dx": dx,
        "dy": dy,
        "time_chunk": time_chunk,
        "min_valid_orientations":
            min_valid_orientations,
    }

    for name in component_names:

        # All 8 signed directions
        result[
            f"direction_{name}"
        ] = direction_values[name]

        # Four opposite-direction-paired results
        # before scale interpolation
        result[
            f"orientation_raw_{name}"
        ] = orientation_values_raw[name]

        # Four orientations interpolated to r_requested
        result[
            f"orientation_{name}"
        ] = orientation_values[name]

        # Anisotropy diagnostics
        result[
            f"A_{name}"
        ] = anisotropy[name]["A"]

        result[
            f"orientation_mean_{name}"
        ] = anisotropy[name]["mean"]

        result[
            f"orientation_std_{name}"
        ] = anisotropy[name]["std"]

        result[
            f"orientation_rms_{name}"
        ] = anisotropy[name]["rms"]

        result[
            f"orientation_cancellation_ratio_{name}"
        ] = anisotropy[name][
            "cancellation_ratio"
        ]

        result[
            f"nvalid_orientations_{name}"
        ] = anisotropy[name][
            "nvalid_orientations"
        ]

    # Convenient aliases
    result["A1L"] = result["A_Dl"]
    result["A1T"] = result["A_Dt"]

    result["A2L"] = result["A_Dll"]
    result["A2T"] = result["A_Dtt"]
    result["A2"] = result["A_D2"]

    result["A3L"] = result["A_Dlll"]
    result["A3LTT"] = result["A_Dltt"]
    result["A3"] = result["A_D3"]

    print(
        f"\nCompleted: {progress_name}\n"
        f"  independent orientations: "
        f"{orientation_angles}"
    )

    return result


# ============================================================
# Save as MATLAB file
# ============================================================
def save_directional_result(
    result,
    output_path,
    case_name,
    length_unit
):
    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    mat_data = {
        "case_name": case_name,
        "length_unit": length_unit,
        "definitions": (
            "Dl=<duL>; Dt=<duT>; D1=Dl+Dt; "
            "Dll=<duL^2>; Dtt=<duT^2>; "
            "D2=Dll+Dtt; Dlll=<duL^3>; "
            "Dltt=<duL*duT^2>; D3=Dlll+Dltt"
        ),
        "anisotropy_definition": (
            "A=std_orientation/"
            "(abs(mean_orientation)+rms_orientation)"
        ),
    }

    for key, value in result.items():

        if isinstance(
            value,
            (np.ndarray, list, tuple)
        ):
            mat_data[key] = np.asarray(value)

        elif isinstance(
            value,
            (int, float, bool, np.number)
        ):
            mat_data[key] = np.asarray(value)

        elif isinstance(value, str):
            mat_data[key] = value

    sio.savemat(
        output_path,
        mat_data,
        do_compression=True
    )

    print(
        f"Saved MATLAB file:\n{output_path}"
    )


def u2rho_3d (var_u):
    [N,Mp,L]=var_u.shape
    Lp=L+1
    Lm=L-1
    var_rho=np.zeros((N,Mp,Lp))
    var_rho[:,:,1:-1]=0.5*(var_u[:,:,1:]+var_u[:,:,:-1])
    var_rho[:,:,0]=var_rho[:,:,1]
    var_rho[:,:,-1]=var_rho[:,:,-2]
    return var_rho

def v2rho_3d (var_v):
    [N,M,Lp]=var_v.shape
    Mp=M+1
    Mm=M-1
    var_rho=np.zeros((N,Mp,Lp))
    var_rho[:,1:-1,:]=0.5*(var_v[:,1:,:]+var_v[:,:-1,:])
    var_rho[:,0,:]=var_rho[:,1,:]
    var_rho[:,-1,:]=var_rho[:,-2,:]
    return var_rho


# 1. Read data

In [12]:
data_dir=f'/meddy/simingzhang/Data/RB_iceland_data/'
fname='z_niskin2km_his_hf_depth_500m_grd.0002.nc'
ds=xr.open_dataset(data_dir+f'iceland_wave/'+fname)
fname2='z_niskin2km_his_smooth_depth_500m_grd.0002.nc'
dsnw=xr.open_dataset(data_dir+f'iceland_no_wave/'+fname2)

In [13]:
u_HF=u2rho_3d(np.squeeze(ds['u'].values))
v_HF=v2rho_3d(np.squeeze(ds['v'].values))
u_SM=u2rho_3d(np.squeeze(dsnw['u'].values))
v_SM=v2rho_3d(np.squeeze(dsnw['v'].values))

In [14]:
import os
outpur_dir='/meddy/simingzhang/Data/HIT2D/2dt2'


grid_dir='/meddy/simingzhang/Data/HIT2D/2dt2'

os.chdir(grid_dir)

import scipy.io as sio
from tqdm import tqdm
grid=sio.loadmat('HIT2D_Parameters.mat')
Diag=grid['Diag']
mm=grid['mm']
kk=grid['kk']
u=np.zeros((199,512,512))
v=np.zeros_like(u)

for i in tqdm(range(0,199)):
    fname=f'HIT2D_t_{i+4000}.mat'
    data=sio.loadmat(fname)
    hq=data['hq']
    
    hpsi=Diag*hq;
    hu1=-1j*mm*hpsi;
    hw1=1j*kk*hpsi;
    u[i,:,:]=np.real(np.fft.ifft2(hu1))
    v[i,:,:]=np.real(np.fft.ifft2(hw1))

u_HIT=u
v_HIT=v

100%|██████████████████████████████████████████████████████████| 199/199 [00:04<00:00, 42.96it/s]


# 2. calc A(r)

In [ ]:
# ============================================================
# Direction definitions
# ============================================================
directions_8 = [
    (0, 1),      # 0 degrees: east
    (1, 1),      # 45 degrees: northeast
    (1, 0),      # 90 degrees: north
    (1, -1),     # 135 degrees: northwest
    (0, -1),     # 180 degrees: west
    (-1, -1),    # 225 degrees: southwest
    (-1, 0),     # 270 degrees: south
    (-1, 1),     # 315 degrees: southeast
]


# ============================================================
# HF calculation
# ============================================================
r_requested_km = np.geomspace(
    2.0,
    300.0,
    20
)

HF_directional = calculate_eulerian_directional_SF(
    u=u_HF,
    v=v_HF,
    r_values=r_requested_km * 1000.0,
    dx=1850,
    dy=1850,
    periodic=False,
    directions=directions_8,
    time_chunk=50,
    show_progress=True,
    progress_name="HF directional Eulerian SF",
    min_valid_orientations=3
)


# ============================================================
# Save HF
# ============================================================
output_dir = Path(
    "/meddy/simingzhang/Data/Figure_SF3"
)

save_directional_result(
    result=HF_directional,
    output_path=(
        output_dir
        / "HF_Eulerian_directional_all_orders.mat"
    ),
    case_name="HF",
    length_unit="m"
)

# ============================================================
# SM calculation
# ============================================================
r_requested_km = np.geomspace(
    2.0,
    300.0,
    20
)

SM_directional = calculate_eulerian_directional_SF(
    u=u_SM,
    v=v_SM,
    r_values=r_requested_km * 1000.0,
    dx=1850,
    dy=1850,
    periodic=False,
    directions=directions_8,
    time_chunk=50,
    show_progress=True,
    progress_name="SM directional Eulerian SF",
    min_valid_orientations=3
)


# ============================================================
# Save SM
# ============================================================
output_dir = Path(
    "/meddy/simingzhang/Data/Figure_SF3"
)

save_directional_result(
    result=SM_directional,
    output_path=(
        output_dir
        / "SM_Eulerian_directional_all_orders.mat"
    ),
    case_name="SM",
    length_unit="m"
)


# ============================================================
# HIT calculation
# ============================================================
dx_hit = 2.0 * np.pi / 512.0

r_requested_hit = np.geomspace(
    2.0 * dx_hit,
    np.pi,
    25
)

HIT_directional = calculate_eulerian_directional_SF(
    u=u_HIT,
    v=v_HIT,
    r_values=r_requested_hit,
    dx=dx_hit,
    dy=dx_hit,
    periodic=True,
    directions=directions_8,
    time_chunk=20,
    show_progress=True,
    progress_name="HIT directional Eulerian SF",
    min_valid_orientations=3
)

# ============================================================
# Save HIT
# ============================================================

output_dir = Path(
    "/meddy/simingzhang/Data/Figure_SF3"
)

save_directional_result(
    result=HIT_directional,
    output_path=(
        output_dir
        / "HIT_Eulerian_directional_all_orders.mat"
    ),
    case_name="HIT",
    length_unit="nondimensional"
)

HF directional Eulerian SF
  velocity shape: (2148, 287, 287)
  requested scales: 20
  signed directions: 8
  time chunk: 50
  periodic: False


HF directional Eulerian SF:  41%|█▏ | 2838/6880 [05:49<07:54,  8.52chunk/s, r=1.65e+04, theta=90]